# W3D5 - Benchmark harness (Qwen2.5-7B-Instruct-AWQ on a Colab T4)

Run the cells **in order, once each**. Do not re-run the install or server cells once they succeeded.

**Part A** - the lab: run `bench.py` on `prompts.txt`, find the knee, fill the capacity note, run the green check.
**Part B** - the project: run the same harness on real course content, with the prefix cache OFF, and compute cost.

Files to upload first (Files panel, left): `bench.py`, `prompts.txt`, `verify_cell.py`, `benchmark_payloads.json`, `smoke_test_request.json`.
Names must match exactly (no "(1)" suffix).

## Setup

In [ ]:
# Install vLLM. Only torchaudio is removed (CUDA mismatch); torchvision must stay.
# If Colab shows "Restart session", click it and do NOT re-run this cell.
!pip install vllm httpx
!pip uninstall -y torchaudio

In [ ]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv
!python3 -c "import torch, torchvision, vllm; print('OK', vllm.__version__, torch.__version__, torchvision.__version__)"

In [ ]:
!ls -la bench.py prompts.txt verify_cell.py benchmark_payloads.json smoke_test_request.json
!wc -l prompts.txt

## Part A - the lab (prefix cache ON, the default)

**Before running the sweep, fill your prediction card by hand** (knee guess and target p95 / SLO).

In [ ]:
import subprocess, time, httpx, os

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"
EAGER = False            # set True only if startup fails
PREFIX_CACHING = True    # Part A: default (True)

def ready():
    try:
        return httpx.get("http://localhost:8000/v1/models", timeout=2).status_code == 200
    except Exception:
        return False

if ready():
    print("Server is already running. Skip to the next cell.")
else:
    cmd = ["python3", "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL,
           "--quantization", "awq",
           "--dtype", "half",
           "--max-model-len", "16384",
           "--gpu-memory-utilization", "0.85",
           "--port", "8000"]
    if EAGER:
        cmd.append("--enforce-eager")
    if not PREFIX_CACHING:
        cmd.append("--no-enable-prefix-caching")

    env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")
    p = subprocess.Popen(cmd, stdout=open("vllm.log", "w"),
                         stderr=subprocess.STDOUT, env=env, start_new_session=True)

    for i in range(60):
        if p.poll() is not None:
            print("Server exited early, exit code:", p.returncode)
            print(subprocess.run("grep 'core.py:1374' vllm.log | tail -6 | cut -c60-400",
                                 shell=True, capture_output=True, text=True).stdout)
            break
        if ready():
            print("Server ready after", i * 10, "seconds")
            break
        time.sleep(10)
    else:
        print("Timed out waiting for the server")

In [ ]:
# Smoke test: the model must answer before the sweep starts
!sed -i 's#Qwen/Qwen2.5-7B-Instruct"#Qwen/Qwen2.5-7B-Instruct-AWQ"#' smoke_test_request.json
!curl -s -X POST http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d @smoke_test_request.json | python3 -m json.tool

In [ ]:
# Lab sweep (about 40 min). Run this ONCE. The file name must stay bench_report.json for the green check.
!rm -f bench_report.json
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-7B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts.txt \
  --out bench_report.json

In [ ]:
# Find the knee: highest concurrency whose p95 is still under your SLO.
import json

TARGET_P95_S = None   # <- set your SLO in seconds from the prediction card, e.g. 5.0
if TARGET_P95_S is None:
    raise SystemExit("Set TARGET_P95_S first (your SLO from the prediction card).")

levels = json.load(open("bench_report.json"))["runs"][-1]["levels"]
for L in levels:
    print(f"c={L['concurrency']:>2}  tok/s={L['tokens_per_s']:>7.1f}  "
          f"ttft_p95={L['ttft_p95_s']}  lat_p95={L['latency_p95_s']}  errors={L['errors']}")

under = [L for L in levels if L["latency_p95_s"] is not None and L["latency_p95_s"] <= TARGET_P95_S]
knee = max(under, key=lambda L: L["concurrency"]) if under else None
print("knee:", knee)

if knee and knee["concurrency"] == max(L["concurrency"] for L in levels):
    print("NOTE: the knee is at the top of the sweep (sweep-bounded). "
          "Say so in the note; the stretch run with --concurrency 32 finds the real edge.")

with open("knee.json", "w") as f:
    json.dump({"target_p95_s": TARGET_P95_S,
               "knee_concurrency": knee["concurrency"] if knee else None}, f)

In [ ]:
# Write capacity-note.md. The numbers are computed; the two sentences below are YOURS.
LIMITING_FAMILY_SENTENCE = ""   # compute vs memory vs overhead: which limits the stack at the knee, and the tell
WHY_KNEE_SENTENCE = ""          # in your own words: why report the knee at the SLO, not the peak
ERRORS_EXPLANATION = ""         # only needed if the sweep had request errors

assert LIMITING_FAMILY_SENTENCE.strip() and WHY_KNEE_SENTENCE.strip(), "Write both sentences first."
assert knee, "No level stayed under the target. Explain this in the note instead."

total_errors = sum(L["errors"] for L in levels)
if total_errors:
    assert ERRORS_EXPLANATION.strip(), f"{total_errors} errors in the sweep: explain them in ERRORS_EXPLANATION."

req_per_s = knee["ok"] / knee["wall_s"] if knee.get("wall_s") else 0.0
lines = [
    "# Capacity note (team, one page)",
    "",
    "## The numbers",
    "",
    "- Locked model: Qwen/Qwen2.5-7B-Instruct-AWQ (vLLM 0.29.0, Tesla T4, fp16, max-model-len 16384)",
    f"- Target p95 end-to-end latency (SLO): {TARGET_P95_S} seconds",
    f"- Knee concurrency: {knee['concurrency']}",
    f"- Tokens per second at the knee: {knee['tokens_per_s']}",
    f"- Max sustainable request rate at the target p95: {req_per_s:.2f} req/s",
    "",
    "## The limiting family",
    "",
    "- " + LIMITING_FAMILY_SENTENCE.strip(),
    "",
    "## Why the knee, not the peak",
    "",
    "- " + WHY_KNEE_SENTENCE.strip(),
]
if total_errors:
    lines += ["", "## Request errors", "", "- " + ERRORS_EXPLANATION.strip()]
open("capacity-note.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
print(open("capacity-note.md", encoding="utf-8").read())

In [ ]:
# Green check (needs bench_report.json, knee.json, capacity-note.md). Expected last line: GREEN CHECK: PASS
!python verify_cell.py

In [ ]:
# Save Part A artifacts BEFORE anything else (Colab runtimes vanish)
from google.colab import files
for f_ in ["bench_report.json", "capacity-note.md", "knee.json"]:
    files.download(f_)

## Part B - the project: real course content, prefix cache OFF

The prefix cache makes repeated prompts look free. Real files are new every time, so measure without it.

1. **Runtime -> Restart session** (packages and uploaded files stay; the server stops).
2. Run the "start server (cache OFF)" cell below, then the remaining cells in order.

In [ ]:
import subprocess, time, httpx, os

MODEL = "Qwen/Qwen2.5-7B-Instruct-AWQ"

def ready():
    try:
        return httpx.get("http://localhost:8000/v1/models", timeout=2).status_code == 200
    except Exception:
        return False

if ready():
    print("A server is already running. Restart the session first so the cache-OFF flag takes effect.")
else:
    cmd = ["python3", "-m", "vllm.entrypoints.openai.api_server",
           "--model", MODEL, "--quantization", "awq", "--dtype", "half",
           "--max-model-len", "16384", "--gpu-memory-utilization", "0.85",
           "--no-enable-prefix-caching",
           "--port", "8000"]
    env = dict(os.environ, VLLM_USE_FLASHINFER_SAMPLER="0")
    p = subprocess.Popen(cmd, stdout=open("vllm.log", "w"),
                         stderr=subprocess.STDOUT, env=env, start_new_session=True)
    for i in range(60):
        if p.poll() is not None:
            print("Server exited early, exit code:", p.returncode)
            print(subprocess.run("grep 'core.py:1374' vllm.log | tail -6 | cut -c60-400",
                                 shell=True, capture_output=True, text=True).stdout)
            break
        if ready():
            print("Server ready (prefix cache OFF) after", i * 10, "seconds")
            break
        time.sleep(10)
    else:
        print("Timed out waiting for the server")

In [ ]:
# bench.py reads one prompt per line, so collapse each file's text onto one line
# (this drops line breaks inside each file; mention it in the report).
import json

payloads = json.load(open("benchmark_payloads.json", encoding="utf-8"))
with open("prompts_real.txt", "w", encoding="utf-8") as f:
    for pl in payloads:
        f.write(" ".join(pl["prompt"].split()) + "\n")
print(len(payloads), "real prompts written")

In [ ]:
!python bench.py \
  --base-url http://localhost:8000 \
  --model "Qwen/Qwen2.5-7B-Instruct-AWQ" \
  --concurrency 1,2,4,8,16 \
  --requests-per-level 20 \
  --prompt-file prompts_real.txt \
  --max-tokens 400 \
  --out bench_report_real.json

In [ ]:
# Cost from the real-content run. Do NOT run verify_cell.py on this file.
import json

GPU_HOURLY_COST = 1.80        # $/hr, cloud-equivalent rate
FILES_PER_COURSE = 12         # ASSUMPTION - confirm with the use-case owner
COURSES_PER_MONTH = 200       # ASSUMPTION - confirm with the use-case owner
GPU_HOURS_PER_DAY = 24        # 24 = always-on instance

levels = json.load(open("bench_report_real.json"))["runs"][-1]["levels"]
print(f"{'conc':>4} {'req/s':>7} {'lat_p95':>8} {'$/course':>10} {'$/month (pay-per-use)':>22}")
for L in levels:
    if not L.get("wall_s") or not L["ok"]:
        continue
    rps = L["ok"] / L["wall_s"]
    cost_req = GPU_HOURLY_COST / (rps * 3600)
    cost_course = cost_req * FILES_PER_COURSE
    print(f"{L['concurrency']:>4} {rps:>7.3f} {str(L['latency_p95_s']):>8} "
          f"{cost_course:>10.4f} {cost_course * COURSES_PER_MONTH:>22.2f}")
print(f"\nAlways-on GPU ({GPU_HOURS_PER_DAY} h/day): ${GPU_HOURLY_COST * GPU_HOURS_PER_DAY * 30:,.2f} per month")

In [ ]:
# Clean shutdown, then save Part B artifacts
!pkill -f "[v]llm" ; sleep 5 ; nvidia-smi --query-gpu=memory.used --format=csv
from google.colab import files
for f_ in ["bench_report_real.json", "prompts_real.txt"]:
    files.download(f_)